In [2]:
import os
os.chdir("/Users/abhyudayalohani/Documents/Codex/2026-09-04/you-are-helping-me-scaffold-a")

In [3]:
import pandas as pd
sales = pd.read_parquet("data/smoke_check/sales")
prices = pd.read_parquet("data/smoke_check/prices")

# 1. Does promo actually lift sales?
merged = sales.merge(prices, on=["store_id", "sku_id"], how="left")  # adjust join keys to your schema
print(merged.groupby("is_promo")["units_sold"].mean())
# Expect promo mean noticeably > non-promo mean. If they're within 10%, elasticity isn't wired right.

# 2. Do intermittent SKUs actually have lots of zero-days?
zero_rate = sales.groupby("sku_id")["units_sold"].apply(lambda s: (s == 0).mean())
print(f"SKUs with >50% zero-days: {(zero_rate > 0.5).mean():.1%}")
# Expect ~5%. If it's ~0% or ~50%, the intermittent flag isn't being applied.

# 3. Is the dirtiness really there?
print(f"Duplicate rows: {sales.duplicated().mean():.3%}")     # expect ~0.5%
print(f"Negative units: {(sales['units_sold'] < 0).mean():.3%}")  # expect ~0.2%
print(f"Missing prices: {prices['price'].isna().mean():.3%}")     # expect nonzero

is_promo
False    1.616391
True     1.645892
Name: units_sold, dtype: float64
SKUs with >50% zero-days: 23.0%
Duplicate rows: 0.497%
Negative units: 0.200%
Missing prices: 0.000%


In [4]:
print(len(sales) / sales[["store_id","sku_id","date"]].drop_duplicates().shape[0])

1.0049972602739725
